In [1]:
%load_ext autoreload
%autoreload 2
import moabb
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import P300
from hoda.hoda import HODA,BTTDA
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import matplotlib.pyplot as plt
import seaborn as sns
from moabb.analysis.plotting import paired_plot, meta_analysis_plot, summary_plot
from sklearn.base import BaseEstimator, ClassifierMixin
from toeplitzlda.classification import ToeplitzLDA
from sklearn.svm import SVC
from moabb.analysis.meta_analysis import (  # noqa: E501
    compute_dataset_statistics,
    find_significant_differences,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from matplotlib import style
plt.style.use('default')

In [3]:
tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #bi2013a(),
    #bi2014a(),
    #bi2014b(),
    #bi2015a(),
    #bi2015b(),
    BNCI2014008(),
    #BNCI2014009(),
    #BNCI2015003(),
    #DemonsP300(),
    #EPFLP300(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019()

]
evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda",
    overwrite=True,
    random_state=42,
    n_jobs=-1,
    #data_size=dict(
    #    policy='per_class',
    #    value=[10]
    #),
    #n_perms=[2],
)

In [4]:
import tensorly as tl
tl.set_backend('cupy', local_threadsafe=False)

In [5]:
import numpy as np

class ToeplitzLDAWrapper(BaseEstimator, ClassifierMixin):

    def fit(self, X, y=None):
        n_epochs, n_channels, n_samples = X.shape
        self.tlda_ = ToeplitzLDA(n_channels=n_channels,
                                 data_is_channel_prime=False)
        X = X.reshape(n_epochs, -1)
        return self.tlda_.fit(X, y)

    def decision_function(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.decision_function(X)

    def predict(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict(X)

    def predict_proba(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict_proba(X)


def reshape(X, y=None):
    return X.reshape((X.shape[0],-1))


In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import SVC
from sklearn.feature_selection import SelectPercentile, f_classif
from mne.decoding import Scaler
from hoda.hoda import HODA
import cupy

pipelines = dict()


pipelines['HODA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    HODA(
        max_iter=64,
        rank=None,
        tol=1e-6,
        init ='svd',
        shrinkage=('lw','lw'),
        toeplitz=None,
        obj='diff',
        solver='lanczos',        
        verbose=False,
        taper=False,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    LinearDiscriminantAnalysis(),
)


pipelines['BTTDA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=5,
        hoda_params=dict(
            max_iter=64,
            rank=None,
            tol=1e-6,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='diff',
            solver='lanczos',        
            verbose=False,
            taper=False,
            keep_train_info=False
        ),
    ),
    #SelectPercentile(percentile=50, score_func=f_classif),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

pipelines['tLDA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    ToeplitzLDAWrapper()
)

In [9]:
results = evaluation.process(pipelines)

008-2014-WithinSession:   0%|                                                                                                           | 0/8 [00:07<?, ?it/s]


TypeError: 'list' object cannot be interpreted as an integer

In [ ]:
results

In [ ]:
stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:

_=meta_analysis_plot(stats,  "tLDA","BTTDA")



In [ ]:
_ = paired_plot(results, "tLDA", "BTTDA")
